In [1]:
import pandas as pd

In [2]:
df_price = pd.read_csv('../data/processed/price.csv')
df_pvgis = pd.read_csv('../data/processed/pvgis.csv')
df_meteo = pd.read_csv('../data/processed/meteo.csv')

In [3]:
df_meteo.columns

Index(['Datetime (UTC)', 'temperature_2m', 'wind_speed_10m',
       'wind_direction_10m', 'cloud_cover', 'shortwave_radiation',
       'precipitation'],
      dtype='object')

In [4]:
df_price.columns

Index(['Country', 'Datetime (UTC)', 'Price (EUR/MWhe)'], dtype='object')

In [5]:
df_pvgis.columns

Index(['Datetime (UTC)', 'P', 'Gb(i)', 'Gd(i)', 'Gr(i)', 'H_sun', 'T2m',
       'WS10m'],
      dtype='object')

In [6]:
df_meteo['Datetime (UTC)'] = pd.to_datetime(df_meteo['Datetime (UTC)'])
df_price['Datetime (UTC)'] = pd.to_datetime(df_price['Datetime (UTC)'])
df_pvgis['Datetime (UTC)'] = pd.to_datetime(df_pvgis['Datetime (UTC)'])

print('openmeteo_df sample:', df_meteo['Datetime (UTC)'].iloc[0])
print('price_df sample:', df_price['Datetime (UTC)'].iloc[0])
print('pvgis_df sample:', df_pvgis['Datetime (UTC)'].iloc[0])

openmeteo_df sample: 2023-07-01 00:00:00
price_df sample: 2015-01-01 00:00:00
pvgis_df sample: 2023-01-01 00:00:00


In [7]:
df_pvgis = df_pvgis[(df_pvgis['Datetime (UTC)'] >= '2023-07-01') &
                     (df_pvgis['Datetime (UTC)'] < '2023-10-01')]

In [8]:
df_merged = pd.merge(df_meteo, df_price, on='Datetime (UTC)', how='inner')

In [9]:
df_final = pd.merge(df_merged, df_pvgis, on='Datetime (UTC)', how='inner')

In [10]:
df_final.shape

(2208, 16)

In [11]:
df_final.head()

,Datetime (UTC),temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,Country,Price (EUR/MWhe),P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m
0,2023-07-01 00:00:00,10.5,4.3,265,48,0.0,0.0,Germany,95.72,0.00,0.00,0.00,0.00,0.00,12.85,1.45
1,2023-07-01 01:00:00,10.4,2.5,188,53,0.0,0.0,Germany,89.61,0.00,0.00,0.00,0.00,0.00,11.90,1.45
2,2023-07-01 02:00:00,8.9,3.1,159,100,0.0,0.0,Germany,86.94,0.00,0.00,0.00,0.00,0.00,10.94,1.52
3,2023-07-01 03:00:00,9.0,3.4,162,100,0.0,0.0,Germany,80.18,2.06,0.00,10.17,0.15,2.03,10.32,1.66
4,2023-07-01 04:00:00,9.7,4.0,153,100,9.0,0.0,Germany,76.74,110.40,92.79,56.78,1.25,10.17,10.96,1.72


In [12]:
df_final['Datetime (UTC)'] = pd.to_datetime(df_final['Datetime (UTC)'])
df_final = df_final.set_index('Datetime (UTC)')

df_final['Datetime (UTC)'] = df_final.index
df_final['hour'] = df_final.index.hour
df_final['day_of_week'] = df_final.index.dayofweek
df_final['month'] = df_final.index.month
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)

In [13]:
df_final.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,Country,Price (EUR/MWhe),P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,Datetime (UTC),hour,day_of_week,month,is_weekend
Datetime (UTC),,,,,,,,,,,,,,,,,,,,
2023-07-01 00:00:00,10.5,4.3,265,48,0.0,0.0,Germany,95.72,0.00,0.00,0.00,0.00,0.00,12.85,1.45,2023-07-01 00:00:00,0,5,7,1
2023-07-01 01:00:00,10.4,2.5,188,53,0.0,0.0,Germany,89.61,0.00,0.00,0.00,0.00,0.00,11.90,1.45,2023-07-01 01:00:00,1,5,7,1
2023-07-01 02:00:00,8.9,3.1,159,100,0.0,0.0,Germany,86.94,0.00,0.00,0.00,0.00,0.00,10.94,1.52,2023-07-01 02:00:00,2,5,7,1
2023-07-01 03:00:00,9.0,3.4,162,100,0.0,0.0,Germany,80.18,2.06,0.00,10.17,0.15,2.03,10.32,1.66,2023-07-01 03:00:00,3,5,7,1
2023-07-01 04:00:00,9.7,4.0,153,100,9.0,0.0,Germany,76.74,110.40,92.79,56.78,1.25,10.17,10.96,1.72,2023-07-01 04:00:00,4,5,7,1


In [15]:
df_final = df_final.drop(columns=['Country'])

In [16]:
df_final.to_csv('../data/processed/final_dataset.csv', index=False)